# 02 — EDA: Dissatisfaction Drivers
**Role of this notebook:** 
- Explore the operational factors associated with customer dissatisfaction using exploratory data analysis (EDA).
- The objective is to identify which operational patterns consistently correlate with negative customer reviews, validate business thresholds before dashboard development, and provide evidence for the KPIs and business rules later implemented in Power BI.

**Business question:** Which operational factors most strongly drive customer dissatisfaction on the Olist platform, and where should management prioritize action?

**Dissatisfaction definition:** Throughout this notebook, customer dissatisfaction is defined as the percentage of reviewed orders with a review score of 1 or 2.

In [1]:
import warnings 
from sqlalchemy.exc import SAWarning 

warnings.filterwarnings("ignore", category=SAWarning) 

import pandas as pd
import numpy as np
from sqlalchemy import create_engine

pd.set_option('display.max_columns', 50)

engine = create_engine(
    "mssql+pyodbc://localhost/olist_dissatisfaction?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Cleaned analytical tables produced by the SQL pipeline
analysis_orders = pd.read_sql("SELECT * FROM analysis_orders", engine, parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
customers   = pd.read_sql("SELECT * FROM raw_customers", engine)
sellers     = pd.read_sql("SELECT * FROM raw_sellers", engine)
products    = pd.read_sql("SELECT * FROM raw_products", engine)
order_items = pd.read_sql("SELECT * FROM raw_order_items", engine)
cat_trans   = pd.read_sql("SELECT * FROM raw_category_translation", engine)

analysis_orders["negative"] = analysis_orders["is_negative_review"].astype(bool)

print(f"analysis_orders: {len(analysis_orders):,} rows (already cleaned and scoped by SQL)")

engine.dispose()


analysis_orders: 96,478 rows (already cleaned and scoped by SQL)


## 1. Platform-wide dissatisfaction rate


In [2]:
reviewed = analysis_orders[analysis_orders['review_score'].notna()]
dissat_rate = reviewed['negative'].mean()
print(f'Dissatisfaction rate: {dissat_rate*100:.1f}%  (n reviewed orders = {len(reviewed):,})')


Dissatisfaction rate: 12.8%  (n reviewed orders = 95,832)


## 2. Delivery cliff

### 2.1 Discovery: raw negative rate by delivery-time window
Before defining delivery buckets, the relationship between delivery time and customer dissatisfaction is explored using simple 5-day intervals.
The objective is to observe the overall pattern first, rather than selecting bucket boundaries upfront.

In [3]:
fine_grained = reviewed.groupby(pd.cut(reviewed['delivery_days'], bins=list(range(0, 50, 5)) + [9999]), observed=True).agg(
    orders=('order_id', 'count'),
    negative_rate=('negative', 'mean')
)
fine_grained['negative_rate_pct'] = (fine_grained['negative_rate']*100).round(1)
fine_grained


,orders,negative_rate,negative_rate_pct
delivery_days,,,
"(0, 5]",16637,0.072068,7.2
"(5, 10]",32984,0.083980,8.4
"(10, 15]",22004,0.096301,9.6
"(15, 20]",11249,0.123478,12.3
"(20, 25]",5826,0.192757,19.3
"(25, 30]",2943,0.331634,33.2
"(30, 35]",1639,0.527761,52.8
"(35, 40]",912,0.666667,66.7
"(40, 45]",638,0.722571,72.3


**Observation:** 
Negative review rates increase gradually during the first ~20 delivery days, then accelerate sharply after approximately 25 days.
Rather than a linear trend, the relationship exhibits a clear "delivery cliff", where every additional delay substantially increases dissatisfaction.
This observation motivates the bucket boundaries defined in Section 2.2.

***Note**: A small number of delivered orders (8) have no recorded delivery date in the source data (see `01_data_quality_and_structure.ipynb`, Section 1.1). These rows are explicitly excluded from the delivery-time analysis below because `delivery_days` cannot be computed.*

### 2.2 Bucket definition (result of the discovery above)
Six buckets: **0–7 / 8–14 / 15–21 / 22–25 / 26–35 / 35+ days**. The first four match the natural weekly-ish breakpoints in the flat/early-rising region; the last two split the accelerating tail into "worsening" (26–35) vs. "severe" (35+) rather than lumping everything past 25 days into one bucket, since Section 2.1 shows those two ranges are meaningfully different (~40% vs. ~72% negative rate).

Once validated, the bucket definition was implemented consistently in Power BI to ensure the dashboard reflects the same analytical logic established during EDA.

In [4]:
before_dropna = len(reviewed)
reviewed_with_delivery = reviewed.dropna(subset=['delivery_days']).copy()
print(f"Dropped {before_dropna - len(reviewed_with_delivery)} orders with no delivery date on record "
      f"(delivered but missing order_delivered_customer_date — see notebook 01, Section 1.1). "
      f"{len(reviewed_with_delivery):,} orders remain for bucket analysis.")

bucket_edges = [-1, 7, 14, 21, 25, 35, 10_000]
bucket_labels = ['0-7 days', '8-14 days', '15-21 days', '22-25 days', '26-35 days', '35+ days']
reviewed_with_delivery['delivery_bucket'] = pd.cut(reviewed_with_delivery['delivery_days'], bins=bucket_edges, labels=bucket_labels)

bucket_stats = reviewed_with_delivery.groupby('delivery_bucket', observed=True).agg(
    orders=('order_id', 'count'),
    negative_rate=('negative', 'mean')
).reindex(bucket_labels)
bucket_stats['negative_rate_pct'] = (bucket_stats['negative_rate']*100).round(1)
assert bucket_stats['orders'].sum() == len(reviewed_with_delivery), 'every remaining order should land in exactly one bucket'
bucket_stats


Dropped 8 orders with no delivery date on record (delivered but missing order_delivered_customer_date — see notebook 01, Section 1.1). 95,824 orders remain for bucket analysis.


,orders,negative_rate,negative_rate_pct
delivery_bucket,,,
0-7 days,30550,0.075548,7.6
8-14 days,37775,0.090907,9.1
15-21 days,16054,0.121963,12.2
22-25 days,4322,0.208237,20.8
26-35 days,4582,0.401790,40.2
35+ days,2541,0.720582,72.1


In [5]:
short = bucket_stats.loc['0-7 days', 'negative_rate']
severe = bucket_stats.loc['35+ days', 'negative_rate']
print(f'0-7 days negative rate:  {short*100:.1f}%')
print(f'35+ days negative rate:  {severe*100:.1f}%')
print(f'Multiplier: {severe/short:.1f}x')


0-7 days negative rate:  7.6%
35+ days negative rate:  72.1%
Multiplier: 9.5x


## 3. High-risk seller identification
Threshold: **≥50 orders AND average review score < 3.5**.

*Unlike product categories, a separate negative-rate threshold is unnecessary because sellers meeting this criterion already exhibit consistently high negative review rates.*

In [6]:
item_seller = order_items.merge(analysis_orders[['order_id']], on='order_id')
item_seller = item_seller.merge(reviewed[['order_id','review_score']], on='order_id', how='inner')

seller_stats = item_seller.groupby('seller_id').agg(
    orders=('order_id', 'nunique'),
    avg_score=('review_score', 'mean'),
    negative_rate=('review_score', lambda x: (x <= 2).mean())
)

risky_sellers = seller_stats[(seller_stats['orders'] >= 50) & (seller_stats['avg_score'] < 3.5)]
print(f'High-risk sellers flagged: {len(risky_sellers)}')
print(f'Minimum negative rate among flagged sellers: {risky_sellers["negative_rate"].min()*100:.1f}%')
risky_sellers.sort_values('avg_score').head(10)


High-risk sellers flagged: 18
Minimum negative rate among flagged sellers: 25.3%


,orders,avg_score,negative_rate
seller_id,,,
1ca7077d890b907f89be8c954a02686a,107,2.269841,0.634921
2eb70248d66e0e3ef83659f71b244378,184,2.809278,0.474227
972d0f9cf61b499a4812cf0bfa3ad3c4,79,2.964286,0.440476
a49928bcdf77c55c6d6e05e09a9b4ca5,96,2.971154,0.432692
8e6d7754bc7e0f22c96d255ebda59eba,84,2.992248,0.449612
bbad7e518d7af88a0897397ffdca1979,67,3.048193,0.445783
54965bbe3e4f07ae045b90b0b8541f52,69,3.065789,0.421053
5058e8c1e82653974541e83690655b4a,61,3.080000,0.373333
8444e55c1f13cd5c179851e5ca5ebd00,92,3.156863,0.382353


**Validation:** The minimum negative review rate among all flagged sellers is 25.3%, confirming that an additional negative-rate threshold would not change the results.

## 4. High-risk category identification
Threshold: **≥100 orders AND (negative rate > 20% OR average score < 3.5)**.


In [7]:
item_cat = order_items.merge(products[['product_id','product_category_name']], on='product_id')
item_cat = item_cat.merge(cat_trans, on='product_category_name', how='left')
item_cat = item_cat.merge(analysis_orders[['order_id']], on='order_id')
item_cat = item_cat.merge(reviewed[['order_id','review_score']], on='order_id', how='inner')

cat_stats = item_cat.groupby('product_category_name_english').agg(
    orders=('order_id', 'nunique'),
    avg_score=('review_score', 'mean'),
    negative_rate=('review_score', lambda x: (x <= 2).mean())
)

risky_cats = cat_stats[(cat_stats['orders'] >= 100) &
                        ((cat_stats['negative_rate'] > 0.20) | (cat_stats['avg_score'] < 3.5))]
print(f'High-risk categories flagged: {len(risky_cats)}')
risky_cats.sort_values('negative_rate', ascending=False)


High-risk categories flagged: 4


,orders,avg_score,negative_rate
product_category_name_english,,,
office_furniture,1244,3.516324,0.254534
fashion_male_clothing,105,3.758065,0.250000
fixed_telephony,209,3.757937,0.230159
audio,345,3.837989,0.215084


## 5. Hidden-risk sellers
Sellers that pass the platform-wide screen in Section 3 but show materially worse outcomes for repeat/priority customers specifically. Priority segment proxy here: repeat customers (`frequency >= 2` at the `customer_unique_id` grain).


In [8]:
cust_freq = analysis_orders.merge(customers, on='customer_id').groupby('customer_unique_id')['order_id'].nunique().rename('frequency')

d2 = analysis_orders.merge(customers[['customer_id','customer_unique_id']], on='customer_id')
d2 = d2.merge(cust_freq, left_on='customer_unique_id', right_index=True)
d2['is_priority'] = d2['frequency'] >= 2

d2_sellers = d2.merge(order_items[['order_id','seller_id']], on='order_id')

seller_priority = d2_sellers[d2_sellers['is_priority']].groupby('seller_id').agg(
    priority_orders=('order_id', 'nunique'),
    priority_negative_rate=('negative', 'mean')
)
seller_overall = d2_sellers.groupby('seller_id').agg(
    overall_orders=('order_id', 'nunique'),
    overall_negative_rate=('negative', 'mean')
)

hidden = seller_priority.join(seller_overall, how='inner')
hidden = hidden[(hidden['priority_orders'] >= 5) &
                (hidden['overall_negative_rate'] < 0.20) &
                (hidden['priority_negative_rate'] >= 0.30)]
hidden = hidden.sort_values('priority_negative_rate', ascending=False)
print(f'Hidden-risk sellers identified: {len(hidden)}')
hidden


Hidden-risk sellers identified: 15


,priority_orders,priority_negative_rate,overall_orders,overall_negative_rate
seller_id,,,,
aac29b1b99776be73c3049939652091d,5,0.545455,94,0.145038
1e8b33f18b4f7598d87f5cbee2282cc2,9,0.500000,122,0.136691
c826c40d7b19f62a09e2d7c5e7295ee2,10,0.454545,346,0.153425
0be8ff43f22e456b4e0371b2245e4d01,6,0.428571,156,0.182857
f5a590cf36251cf1162ea35bef76fe84,7,0.428571,113,0.110169
0db783cfcd3b73998abc6e10e59a102f,5,0.400000,132,0.124088
f4aba7c0bca51484c30ab7bdc34bcdd1,5,0.400000,108,0.173913
522620dcb18a6b31cd7bdf73665113a9,8,0.375000,172,0.185393
a673821011d0cec28146ea42f5ab767f,10,0.363636,125,0.184397


## 6. Geographic drivers: delivery & satisfaction by state
`is_late` and `delay_days` are already computed in SQL — this section only aggregates by state.


In [9]:
geo = reviewed.merge(customers[['customer_id','customer_state']], on='customer_id')

state_stats = geo.groupby('customer_state').agg(
    delivered_orders=('order_id', 'count'),
    avg_review_score=('review_score', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    late_delivery_rate=('is_late', 'mean'),
    avg_delay_days=('delay_days', lambda x: x[x > 0].mean())
).round(2)

state_stats.sort_values('avg_review_score').head(10)


,delivered_orders,avg_review_score,avg_delivery_days,late_delivery_rate,avg_delay_days
customer_state,,,,,
MA,712,3.83,21.38,0.19,10.52
AL,394,3.85,24.41,0.23,9.68
RR,41,3.90,29.34,0.12,36.40
PA,933,3.91,23.59,0.12,12.89
SE,334,3.91,21.38,0.15,16.40
BA,3229,3.93,19.19,0.14,11.86
CE,1273,3.94,21.14,0.15,15.05
RJ,12211,3.97,15.17,0.13,13.47
PI,471,3.99,19.43,0.16,13.48


In [10]:
north_northeast = ['AC','AP','AM','PA','RO','RR','TO','AL','BA','CE','MA','PB','PE','PI','RN','SE']
state_stats['region'] = np.where(state_stats.index.isin(north_northeast), 'North/Northeast', 'Other')

region_summary = state_stats.groupby('region').agg(
    states=('avg_review_score', 'count'),
    avg_review_score=('avg_review_score', 'mean'),
    avg_delivery_days=('avg_delivery_days', 'mean'),
    late_delivery_rate=('late_delivery_rate', 'mean')
).round(2)
print(region_summary)
print()
print('SP (best-performing state):')
print(state_stats.loc['SP'])


                 states  avg_review_score  avg_delivery_days  \
region                                                         
North/Northeast      16              4.03              21.81   
Other                11              4.14              14.10   

                 late_delivery_rate  
region                               
North/Northeast                0.12  
Other                          0.08  

SP (best-performing state):
delivered_orders      40273
avg_review_score       4.25
avg_delivery_days      8.69
late_delivery_rate     0.06
avg_delay_days         8.32
region                Other
Name: SP, dtype: object


**Finding:** North and Northeast states consistently experience longer delivery times, higher late-delivery rates, and lower review scores than the rest of the country.
**São Paulo (SP)** shows the strongest delivery performance across all major delivery metrics, making it a useful operational benchmark.


## 7. Counterfactual: dissatisfaction rate if risk factors were resolved
Scenario analysis (not causal inference): This section estimates how the dissatisfaction rate could change if orders exposed to known operational risk factors performed similarly to orders without those risks.
The objective is to estimate potential business impact rather than establish causality.

In [11]:
analysis_orders_r = reviewed.copy()
# delivery_risk flags the two worst buckets from Section 2 (26-35 and 35+ days combined)
analysis_orders_r['delivery_risk'] = analysis_orders_r['delivery_days'] > 25

order_sellers = order_items.groupby('order_id')['seller_id'].apply(set)
analysis_orders_r['seller_risk'] = analysis_orders_r['order_id'].map(lambda oid: bool(order_sellers.get(oid, set()) & set(risky_sellers.index)))

order_cats = item_cat.groupby('order_id')['product_category_name_english'].apply(set)
analysis_orders_r['category_risk'] = analysis_orders_r['order_id'].map(lambda oid: bool(order_cats.get(oid, set()) & set(risky_cats.index)))

analysis_orders_r['any_risk'] = analysis_orders_r[['delivery_risk','seller_risk','category_risk']].any(axis=1)

share_touched = analysis_orders_r['any_risk'].mean()
baseline_negrate = analysis_orders_r.loc[~analysis_orders_r['any_risk'], 'negative'].mean()
actual_negrate = analysis_orders_r['negative'].mean()

print(f'Share of orders touched by >=1 risk factor: {share_touched*100:.1f}%')
print(f'Negative rate, no risk factor present (baseline): {baseline_negrate*100:.2f}%')
print(f'Actual overall negative rate: {actual_negrate*100:.2f}%')
print(f'Combined counterfactual: {baseline_negrate*100:.2f}%')

baseline_no_delivery_risk = analysis_orders_r.loc[~analysis_orders_r['delivery_risk'], 'negative'].mean()
print(f'Delivery-only counterfactual: {baseline_no_delivery_risk*100:.2f}%')


Share of orders touched by >=1 risk factor: 10.7%
Negative rate, no risk factor present (baseline): 9.23%
Actual overall negative rate: 12.81%
Combined counterfactual: 9.23%
Delivery-only counterfactual: 9.70%


> **Scenario interpretation:** Resolving the identified operational risk factors could reduce the customer dissatisfaction rate from **12.81%** to approximately **9.23%**, while resolving delivery-related issues alone could reduce it to around **9.70%**. These estimates are directional and are intended to illustrate the potential business impact rather than predict exact outcomes.

## Summary of findings
| Driver | Finding | Report page |
|---|---|---|
| Delivery time | Non-linear "cliff": negative rate accelerates past ~25 days, reaching ~9.5x the 0-7 day rate by 35+ days — basis for the 6-bucket delivery scheme (0-7/8-14/15-21/22-25/26-35/35+) | Delivery Performance |
| Seller risk | ~18–19 sellers meet the ≥50-order / <3.5-score threshold | Seller & Product Quality |
| Category risk | 4 categories meet the ≥100-order / >20% negative or <3.5-score threshold | Seller & Product Quality |
| Hidden-risk sellers | A small set of sellers pass platform-wide screening but show materially worse outcomes for repeat/priority customers | Seller & Product Quality |
| Geography | North/Northeast states underperform on delivery speed, lateness, and review score vs. SP and the rest of the country | Delivery Performance |
| Counterfactual | Resolving known risk factors would bring dissatisfaction down toward the ~9% range | Recommendations |

Next: `03_segmentation_validation.ipynb` — why RFM was abandoned, and K-Means cross-validation of the rule-based customer segmentation.
